<a href="https://colab.research.google.com/github/thakurakanksha288/AI-ASSIGNMENT-PYTHON-/blob/main/1024160019_assignment10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The F1 Mission: Real-Time Telemetry Diagnostics
In Formula 1, the gearbox is the most stressed component of the car. We are monitoring the High-Frequency Vibration Sensor located on the transmission casing.

The Two Failure Modes we must solve:
The "Instant DNF" (Spatial Problem): A catastrophic gear tooth shear. This creates a unique "jagged" signature in the vibration wave that wasn't there before.

The "Thermal Fatigue" (Temporal Problem): As the oil thins and heat rises, the gears don't engage as crisp as they did in Lap 1. The vibration "beat" slows down by milliseconds—invisible to humans, but detectable by AI.

In [6]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Conv1D, Conv1DTranspose, RepeatVector, TimeDistributed, Dropout

# --- GENERATE TELEMETRY ---
t = np.linspace(0, 100, 2000)
# Normal engine harmonics + sensor noise
telemetry = np.sin(t * 5) + 0.5 * np.sin(t * 12) + np.random.normal(0, 0.07, 2000)

# FAILURE 1: Mechanical Impact (Lap 15) - Sharp, local distortion
telemetry[600:610] += 3.5

# FAILURE 2: Thermal Decay (Lap 40 onwards) - Rhythmic slowing
# The frequency shifts from 5.0 to 4.2
telemetry[1400:1800] = np.sin(t[1400:1800] * 4.2) + 0.5 * np.sin(t[1400:1800] * 10)

# --- PREPROCESSING ---
scaler = MinMaxScaler()
scaled_telemetry = scaler.fit_transform(telemetry.reshape(-1, 1))

def create_windows(data, window_size=30):
    return np.array([data[i:i+window_size] for i in range(len(data)-window_size)])

X = create_windows(scaled_telemetry)
# X shape: (Samples, 30 Time_Steps, 1 Feature)

The CNN works like a facial recognition system for data. It learns the "face" of a healthy vibration. When the "Mechanical Impact" occurs, the CNN sees a "deformed face" in its 30-step window and flags it.

In [7]:
model_cnn = Sequential([
    Conv1D(64, kernel_size=5, padding='same', activation='relu', input_shape=(30, 1)),
    Conv1D(32, kernel_size=3, padding='same', activation='relu'),
    Conv1DTranspose(32, kernel_size=3, padding='same', activation='relu'),
    Conv1DTranspose(64, kernel_size=5, padding='same', activation='relu'),
    TimeDistributed(Dense(1))
])
model_cnn.compile(optimizer='adam', loss='mse')
model_cnn.fit(X, X, epochs=100, verbose=0)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



The LSTM doesn't just look at the shape; it listens to the tempo. It has a "Cell State" (internal memory) that keeps track of the rhythm. When the frequency drifts during "Thermal Decay," the LSTM notices that the timing is off.

In [8]:
model_lstm = Sequential([
    LSTM(64, activation='tanh', input_shape=(30, 1), return_sequences=False),
    RepeatVector(30),
    LSTM(64, activation='tanh', return_sequences=True),
    TimeDistributed(Dense(1))
])
model_lstm.compile(optimizer='adam', loss='mse')
model_lstm.fit(X, X, epochs=100, verbose=0)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



In [9]:
# Predictions
cnn_reconstruction = model_cnn.predict(X)
lstm_reconstruction = model_lstm.predict(X)

# Error Calculation (MAE)
cnn_mae = np.mean(np.abs(cnn_reconstruction - X), axis=1).flatten()
lstm_mae = np.mean(np.abs(lstm_reconstruction - X), axis=1).flatten()

# Visualization
fig = go.Figure()
fig.add_trace(go.Scatter(y=telemetry, name="Raw Telemetry", line=dict(color='rgba(255,255,255,0.2)')))
fig.add_trace(go.Scatter(y=cnn_mae, name="CNN Score (Impact Detection)", line=dict(color='cyan', width=2)))
fig.add_trace(go.Scatter(y=lstm_mae, name="LSTM Score (Fatigue Detection)", line=dict(color='magenta', width=2)))

fig.update_layout(
    title="F1 Mission Control: CNN vs. LSTM Diagnostic Battle",
    xaxis_title="Time Steps",
    yaxis_title="Anomaly Score",
    template='plotly_dark'
)
fig.show()

62/62 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step
62/62 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step


Why did the CNN spike early?
The CNN is a Feature Extractor. Its filters are trained to recognize the specific peaks and valleys of a healthy gear mesh. When the "Mechanical Snap" occurs, the data points deviate vertically. The CNN detects this Spatial Distortion immediately. It is the perfect "Ouch!" sensor.

Why did the LSTM catch the slow drift?
The LSTM is a Sequence Predictor. It builds an internal clock. When we slowed the frequency (Thermal Fatigue), the data points stayed within "normal" ranges (0 to 1), but they arrived at the wrong time. The CNN is "fooled" because the shape looks okay, but the LSTM’s memory is "insulted" because the rhythm is broken.

In the high-stakes environment of Formula 1, 'good enough' doesn't exist. This project demonstrates that one model is never enough for total safety. By pitting a CNN against an LSTM, we have created a dual-layered defense system. The CNN provides 'Visual Reflexes' for sudden impacts, while the LSTM provides 'Cognitive Memory' for slow, systemic decay. Mastering both proves that you aren't just a coder,you are a Digital Strategist capable of protecting multi-million dollar assets with the power of Deep Learning.

In [10]:
import plotly.graph_objects as go

fig_raw = go.Figure()

# Plotting the raw telemetry signal
fig_raw.add_trace(go.Scatter(
    y=telemetry,
    mode='lines',
    name='Vibration Sensor',
    line=dict(color='#00ffce', width=1)
))

# Annotating the 'Mechanical Snap'
fig_raw.add_annotation(x=605, y=4, text="Impact Event", showarrow=True, arrowhead=1, bgcolor="#ff0000")

# Annotating the 'Thermal Fatigue'
fig_raw.add_annotation(x=1600, y=1.5, text="Thermal Decay Zone", showarrow=True, arrowhead=1, bgcolor="#ab63fa")

fig_raw.update_layout(
    title="F1 Gearbox Raw Telemetry (High-Frequency Vibration)",
    template='plotly_dark',
    xaxis_title="Time Steps (ms)",
    yaxis_title="Amplitude (G-Force)"
)
fig_raw.show()

In [11]:
# Normalizing errors for visual comparison
cnn_norm = (cnn_mae - np.min(cnn_mae)) / (np.max(cnn_mae) - np.min(cnn_mae))
lstm_norm = (lstm_mae - np.min(lstm_mae)) / (np.max(lstm_mae) - np.min(lstm_mae))

fig_ai = go.Figure()

# CNN Error Trace
fig_ai.add_trace(go.Scatter(
    y=cnn_norm,
    name='CNN (Spatial Reflex)',
    line=dict(color='cyan', width=2),
    fill='tozeroy',
    fillcolor='rgba(0, 255, 255, 0.1)'
))

# LSTM Error Trace
fig_ai.add_trace(go.Scatter(
    y=lstm_norm,
    name='LSTM (Temporal Memory)',
    line=dict(color='magenta', width=2),
    fill='tozeroy',
    fillcolor='rgba(255, 0, 255, 0.1)'
))

fig_ai.update_layout(
    title="Battle of the Architectures: Detection Sensitivity",
    template='plotly_dark',
    xaxis_title="Lap Progress",
    yaxis_title="Anomaly Confidence Score (0-1)",
    hovermode="x unified"
)
fig_ai.show()

What it represents: This is the unprocessed vibration data from the gearbox housing.

The "Spike": At step 600, a massive amplitude jump signifies a mechanical impact (gear tooth shear). This is a spatial anomaly.

The "Drift": From step 1400 onwards, the frequency slows down. The waves are still "clean," but they are stretched out. This is a temporal anomaly.

Narrator's Note: This graph shows why a single alarm system fails. If you set a threshold for the "Spike," you will completely miss the "Drift," which can be just as fatal for the car.

The Insight: The CNN error is represented by a sharp, aggressive spike at the exact moment of mechanical failure.

Why it works: Because the CNN uses Convolutional Filters, it acts like a magnifying glass looking for specific "jagged" shapes. It doesn't care about the history of the race; it only sees that the current 30ms window looks "broken."

Result: It provides instantaneous detection of catastrophic physical damage.

The Insight: While the CNN is quiet during the slow frequency shift, the LSTM error begins a steady, rhythmic climb.

Why it works: The LSTM maintains a Cell State (long-term memory). It remembers the high-speed "beat" of the gearbox from Lap 1. When the friction causes the gearbox to slow down, the LSTM’s "internal prediction" for the next wave is wrong.

Result: It detects systemic fatigue and thermal decay that a spatial model (CNN) would ignore as "normal noise."

The "Mark-Leaving" Moment: By overlaying both scores, we create a Diagnostic Heatmap.

Zone A (CNN High, LSTM Low): Diagnosis: Sudden mechanical failure. Action: Box immediately.

Zone B (CNN Low, LSTM High): Diagnosis: Thermal degradation/Wear. Action: Adjust engine mapping to save the gearbox.

These graphs prove that intelligence in F1 is about perspective. The CNN provides the 'Eyes'—catching the immediate physical breaks. The LSTM provides the 'Brain'—feeling the change in the rhythm of the car. Together, they form a complete protective shield, ensuring the driver is alerted to a failing gearbox whether it happens in a millisecond or over an entire race distance